In [1]:
getwd()
setwd("/liulab/galib/dlbcl_manuscript/")
library(tidyverse)
library(Seurat)
library(ktplots)
library(ComplexHeatmap)
library(circlize)
library(Polychrome)
library(RColorBrewer)
library(pals)
source('./scripts/scplot.R')

[1] "/liulab/galib/dlbcl_manuscript/scripts"

Warning message:
“package ‘tidyverse’ was built under R version 4.1.3”
── Attaching packages ─────────────────────────────────────── tidyverse 1.3.1 ──

✔ ggplot2 3.3.6      ✔ purrr   0.3.4 
✔ tibble  3.1.8      ✔ dplyr   1.0.10
✔ tidyr   1.2.0      ✔ stringr 1.4.1 
✔ readr   2.1.2      ✔ forcats 0.5.1 

Warning message:
“package ‘tidyr’ was built under R version 4.1.2”
Warning message:
“package ‘readr’ was built under R version 4.1.2”
Warning message:
“package ‘forcats’ was built under R version 4.1.3”
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()

Attaching SeuratObject

Attaching sp

Warning message:
“package ‘ComplexHeatmap’ was built under R version 4.1.3”
Loading required package: grid

ComplexHeatmap version 2.10.0
Bioconductor page: http://bioconductor.org/packages/ComplexHeatmap/
Github page: https://github.com/jokergoo/ComplexHeatmap
Documentation: http://jokergoo.g

## reformat data to input of cellphoneDB

In [ ]:
merged<- readRDS("./data/objects/merged_res1.5_obj.rds")
cd3_pos_cd4_neg<- readRDS("./data/objects/cd3_pos_cd4_neg_final.rds")
cd3_pos_cd8_neg<- readRDS("./data/objects/cd3_pos_cd8_neg_final.rds")
B_cell<- readRDS("./data/objects/B_cell_final.rds")

In [ ]:
# convert mouse gene name to human gene name
count_tbl<- merged@assays$RNA@counts
all_genes<- rownames(count_tbl)
mm_hs_genes<- readRDS("./data/cellphoneDB/GeneID_Annotation_mmu_hsa.rds")
mm_hs_genes$mmu_symbol<- mm_hs_genes$mmu_symbol  %>%  str_to_title()
mm_hs_genes<- mm_hs_genes[!mm_hs_genes$mmu_symbol  %>% duplicated(),]  %>% filter(hsa_symbol != "")
shared_genes<- intersect(mm_hs_genes$mmu_symbol, all_genes)
shared_genes  %>% length()
rownames(mm_hs_genes)<- mm_hs_genes$mmu_symbol
table(rownames(count_tbl)== "")
new_count_tbl<- count_tbl[shared_genes,]
rownames(new_count_tbl)<- mm_hs_genes[shared_genes, "hsa_symbol"]
rownames(new_count_tbl)  %>% duplicated()  %>% table()
expr_sum<- rowSums(new_count_tbl)
df<- data.frame(exprSum = expr_sum, idx = c(1:length(expr_sum)), genes = names(expr_sum))  %>% arrange(-exprSum)
deduplicated_count<- new_count_tbl[df[!df$genes  %>% duplicated(),"idx"],]
deduplicated_count  %>% dim()
rownames(deduplicated_count)<- rownames(deduplicated_count)  %>% toupper()
cellphoneDB_obj <- CreateSeuratObject(counts = deduplicated_count, project = "cellphoneDB")

In [ ]:
cd3_pos_cd4_neg<- readRDS("./data/objects/cd3_pos_cd4_neg_final.rds")
cd3_pos_cd8_neg<- readRDS("./data/objects/cd3_pos_cd8_neg_final.rds")
non_B_non_T<- readRDS("./data/objects/non_B_non_T_c18_subclustered.rds")
B_cell<- readRDS("./data/objects/B_cell_final.rds")

## Merge clusters with similar celltypes

In [ ]:
# B cell space
B_cell@meta.data[B_cell@meta.data  %>% filter(seurat_clusters %in% c(0,6,9,10,11,12,13,14,24))  %>% rownames(), "merged_annotation"] = "Follicular"
B_cell@meta.data[B_cell@meta.data  %>% filter(seurat_clusters %in% c(1))  %>% rownames(), "merged_annotation"] = "Marginal"
B_cell@meta.data[B_cell@meta.data  %>% filter(seurat_clusters %in% c(7,8))  %>% rownames(), "merged_annotation"] = "Age-associated/autoimmune"
B_cell@meta.data[B_cell@meta.data  %>% filter(seurat_clusters %in% c(4,5))  %>% rownames(), "merged_annotation"] = "Atypical"
B_cell@meta.data[B_cell@meta.data  %>% filter(seurat_clusters %in% c(3))  %>% rownames(), "merged_annotation"] = "Transcriptionally active"
B_cell@meta.data[B_cell@meta.data  %>% filter(seurat_clusters %in% c(18,19,21))  %>% rownames(), "merged_annotation"] = "Cycling"
B_cell@meta.data[B_cell@meta.data  %>% filter(seurat_clusters %in% c(15))  %>% rownames(), "merged_annotation"] = "Plasmablasts"
B_cell@meta.data[B_cell@meta.data  %>% filter(seurat_clusters %in% c(22))  %>% rownames(), "merged_annotation"] = "Post-GC"
B_cell@meta.data$annotation <- as.character(B_cell@meta.data$annotation)
B_cell@meta.data[is.na(B_cell@meta.data$merged_annotation), "merged_annotation"] = "exclude"

B_cell$merged_annotation  %>% table()
DimPlot(B_cell, group.by = "merged_annotation")
ggsave("./results/figures/9_B_cell_merged_cluster_annotation.pdf", width = 8, height = 6)

# CD8 space
cd3_pos_cd4_neg@meta.data[cd3_pos_cd4_neg@meta.data  %>% filter(seurat_clusters %in% c(0,4))  %>% rownames(), "merged_annotation"] = "Naïve"
cd3_pos_cd4_neg@meta.data[cd3_pos_cd4_neg@meta.data  %>% filter(seurat_clusters %in% c(1,6,7))  %>% rownames(), "merged_annotation"] = "Tcm/Tem"
cd3_pos_cd4_neg@meta.data[cd3_pos_cd4_neg@meta.data  %>% filter(seurat_clusters %in% c(2,3,8,13,16))  %>% rownames(), "merged_annotation"] = "CTLs"
cd3_pos_cd4_neg@meta.data[cd3_pos_cd4_neg@meta.data  %>% filter(seurat_clusters %in% c(11))  %>% rownames(), "merged_annotation"] = "Cycling"
cd3_pos_cd4_neg$annotation <- as.character(cd3_pos_cd4_neg$annotation)
cd3_pos_cd4_neg@meta.data[is.na(cd3_pos_cd4_neg@meta.data$merged_annotation), "merged_annotation"] = "exclude"

cd3_pos_cd4_neg$merged_annotation  %>% table()
DimPlot(cd3_pos_cd4_neg, group.by = "merged_annotation")
ggsave("./results/figures/9_cd3_pos_cd4_neg_merged_cluster_annotation.pdf", width = 8, height = 6)

# CD4 space
cd3_pos_cd8_neg@meta.data[cd3_pos_cd8_neg@meta.data  %>% filter(seurat_clusters %in% c(0,'4_b'))  %>% rownames(), "merged_annotation"] = "Naïve"
cd3_pos_cd8_neg@meta.data[cd3_pos_cd8_neg@meta.data  %>% filter(seurat_clusters %in% c(2))  %>% rownames(), "merged_annotation"] = "Th1/Tem"
cd3_pos_cd8_neg@meta.data[cd3_pos_cd8_neg@meta.data  %>% filter(seurat_clusters %in% c(1,'4_a',6,13))  %>% rownames(), "merged_annotation"] = "CTLs"
cd3_pos_cd8_neg@meta.data[cd3_pos_cd8_neg@meta.data  %>% filter(seurat_clusters %in% c(5))  %>% rownames(), "merged_annotation"] = "Tfh"
cd3_pos_cd8_neg@meta.data[cd3_pos_cd8_neg@meta.data  %>% filter(seurat_clusters %in% c(3,10))  %>% rownames(), "merged_annotation"] = "Tregs"
cd3_pos_cd8_neg@meta.data[cd3_pos_cd8_neg@meta.data  %>% filter(seurat_clusters %in% c(7))  %>% rownames(), "merged_annotation"] = "Follicular Tregs"
cd3_pos_cd8_neg@meta.data[cd3_pos_cd8_neg@meta.data  %>% filter(seurat_clusters %in% c(11,12,17,24))  %>% rownames(), "merged_annotation"] = "Cycling"
cd3_pos_cd8_neg$annotation <- as.character(cd3_pos_cd8_neg$annotation)
cd3_pos_cd8_neg@meta.data[is.na(cd3_pos_cd8_neg@meta.data$merged_annotation), "merged_annotation"] = "exclude"

cd3_pos_cd8_neg$merged_annotation  %>% table()
DimPlot(cd3_pos_cd8_neg, group.by = "merged_annotation")
ggsave("./results/figures/9_cd3_pos_cd8_neg_merged_cluster_annotation.pdf", width = 8, height = 6)

In [ ]:
cellphoneDB_obj$annotation = NA
cellphoneDB_obj@meta.data[rownames(cd3_pos_cd4_neg@meta.data), "annotation"] = as.character(cd3_pos_cd4_neg$new_annotation)
cellphoneDB_obj@meta.data[rownames(cd3_pos_cd8_neg@meta.data), "annotation"] = as.character(cd3_pos_cd8_neg$new_annotation)
cellphoneDB_obj@meta.data[rownames(B_cell@meta.data), "annotation"] = as.character(B_cell$new_annotation)

cellphoneDB_obj@meta.data[rownames(cd3_pos_cd4_neg@meta.data), "space"] = "cd3_pos_cd4_neg"
cellphoneDB_obj@meta.data[rownames(cd3_pos_cd8_neg@meta.data), "space"] = "cd3_pos_cd8_neg"
cellphoneDB_obj@meta.data[rownames(B_cell@meta.data), "space"] = "B_cell"

cellphoneDB_obj@meta.data[rownames(cd3_pos_cd4_neg@meta.data), "genotype"] = cd3_pos_cd4_neg$genotype
cellphoneDB_obj@meta.data[rownames(cd3_pos_cd8_neg@meta.data), "genotype"] = cd3_pos_cd8_neg$genotype
cellphoneDB_obj@meta.data[rownames(B_cell@meta.data), "genotype"] = B_cell$genotype

cellphoneDB_obj@meta.data[rownames(cd3_pos_cd4_neg@meta.data), "age"] = cd3_pos_cd4_neg$age
cellphoneDB_obj@meta.data[rownames(cd3_pos_cd8_neg@meta.data), "age"] = cd3_pos_cd8_neg$age
cellphoneDB_obj@meta.data[rownames(B_cell@meta.data), "age"] = B_cell$age


cellphoneDB_obj@meta.data[rownames(cd3_pos_cd4_neg@meta.data), "pool_id"] = cd3_pos_cd4_neg$pool_id
cellphoneDB_obj@meta.data[rownames(cd3_pos_cd8_neg@meta.data), "pool_id"] = cd3_pos_cd8_neg$pool_id
cellphoneDB_obj@meta.data[rownames(B_cell@meta.data), "pool_id"] = B_cell$pool_id

cellphoneDB_obj@meta.data[rownames(cd3_pos_cd4_neg@meta.data), "merged_annotation"] = cd3_pos_cd4_neg$merged_annotation
cellphoneDB_obj@meta.data[rownames(cd3_pos_cd8_neg@meta.data), "merged_annotation"] = cd3_pos_cd8_neg$merged_annotation
cellphoneDB_obj@meta.data[rownames(B_cell@meta.data), "merged_annotation"] = B_cell$merged_annotation

cellphoneDB_obj@meta.data[is.na(cellphoneDB_obj@meta.data$merged_annotation), "merged_annotation"] = "exclude"

cellphoneDB_obj$new_annotation<- paste0(cellphoneDB_obj@meta.data$space, "_",cellphoneDB_obj@meta.data$merged_annotation)

cellphoneDB_final<- subset(cellphoneDB_obj, merged_annotation != "exclude")
saveRDS(object = cellphoneDB_final, file = "./data/cellphoneDB/cellphoneDB_final.obj")

## obtain 10X formated count table to run cellphonedb

In [ ]:
cellphoneDB_bcl6_6mos<- cellphoneDB_final@meta.data  %>% filter(genotype == "Bcl6tg/+")  %>% filter(age == "6mos")  %>% rownames()
cellphoneDB_bcl6_14mos<- cellphoneDB_final@meta.data  %>% filter(genotype == "Bcl6tg/+")  %>% filter(age == "14mos")  %>% rownames()
cellphoneDB_bcl6_18mos<- cellphoneDB_final@meta.data  %>% filter(genotype == "Bcl6tg/+")  %>% filter(age == "18mos")  %>% rownames()
cellphoneDB_bcl6_sick<- cellphoneDB_final@meta.data  %>% filter(genotype == "Bcl6tg/+")  %>% filter(age == "sick")  %>% rownames()

cellphoneDB_bcl6_6mos_obj<- subset(cellphoneDB_final, cells = cellphoneDB_bcl6_6mos)
cellphoneDB_bcl6_14mos_obj<- subset(cellphoneDB_final, cells = cellphoneDB_bcl6_14mos)
cellphoneDB_bcl6_18mos_obj<- subset(cellphoneDB_final, cells = cellphoneDB_bcl6_18mos)
cellphoneDB_bcl6_sick_obj<- subset(cellphoneDB_final, cells = cellphoneDB_bcl6_sick)

saveRDS(object = cellphoneDB_bcl6_6mos_obj, file = "./data/cellphoneDB/cellphoneDB_bcl6_6mos.obj")
saveRDS(object = cellphoneDB_bcl6_14mos_obj, file = "./data/cellphoneDB/cellphoneDB_bcl6_14mos.obj")
saveRDS(object = cellphoneDB_bcl6_18mos_obj, file = "./data/cellphoneDB/cellphoneDB_bcl6_18mos.obj")
saveRDS(object = cellphoneDB_bcl6_sick_obj, file = "./data/cellphoneDB/cellphoneDB_bcl6_sick.obj")

write10xCounts(x = cellphoneDB_bcl6_6mos_obj@assays$RNA@counts, path = "./data/cellphoneDB/10X_format/bcl6_6mos/", version = '3')
write10xCounts(x = cellphoneDB_bcl6_14mos_obj@assays$RNA@counts, path = "./data/cellphoneDB/10X_format/bcl6_14mos/", version = '3')
write10xCounts(x = cellphoneDB_bcl6_18mos_obj@assays$RNA@counts, path = "./data/cellphoneDB/10X_format/bcl6_18mos/", version = '3')
write10xCounts(x = cellphoneDB_bcl6_sick_obj@assays$RNA@counts, path = "./data/cellphoneDB/10X_format/bcl6_sick", version = '3')

cellphoneDB_bcl6_6mos_meta<- data.frame(Cell = rownames(cellphoneDB_bcl6_6mos_obj@meta.data), 
                                        cell_type = cellphoneDB_bcl6_6mos_obj@meta.data$new_annotation)
cellphoneDB_bcl6_14mos_meta<- data.frame(Cell = rownames(cellphoneDB_bcl6_14mos_obj@meta.data), 
                                        cell_type = cellphoneDB_bcl6_14mos_obj@meta.data$new_annotation)

cellphoneDB_bcl6_18mos_meta<- data.frame(Cell = rownames(cellphoneDB_bcl6_18mos_obj@meta.data), 
                                        cell_type = cellphoneDB_bcl6_18mos_obj@meta.data$new_annotation)

cellphoneDB_bcl6_sick_meta<- data.frame(Cell = rownames(cellphoneDB_bcl6_sick_obj@meta.data), 
                                        cell_type = cellphoneDB_bcl6_sick_obj@meta.data$new_annotation)

write.table(cellphoneDB_bcl6_6mos_meta, file = "./data/cellphoneDB/10X_format/bcl6_6mos_meta.txt", sep = "\t", 
            col.names = TRUE, row.names = FALSE, quote = FALSE)
write.table(cellphoneDB_bcl6_14mos_meta, file = "./data/cellphoneDB/10X_format/bcl6_14mos_meta.txt", sep = "\t", 
            col.names = TRUE, row.names = FALSE, quote = FALSE)
write.table(cellphoneDB_bcl6_18mos_meta, file = "./data/cellphoneDB/10X_format/bcl6_18mos_meta.txt", sep = "\t", 
            col.names = TRUE, row.names = FALSE, quote = FALSE)
write.table(cellphoneDB_bcl6_sick_meta, file = "./data/cellphoneDB/10X_format/bcl6_sick_meta.txt", sep = "\t", 
            col.names = TRUE, row.names = FALSE, quote = FALSE)

In [ ]:
Genotype = "CD70-/-;Bcl6tg/+"

cellphoneDB_bcl6_double_6mos<- cellphoneDB_final@meta.data  %>% filter(genotype == Genotype)  %>% filter(age == "6mos")  %>% rownames()
cellphoneDB_bcl6_double_14mos<- cellphoneDB_final@meta.data  %>% filter(genotype == Genotype)  %>% filter(age == "14mos")  %>% rownames()
cellphoneDB_bcl6_double_18mos<- cellphoneDB_final@meta.data  %>% filter(genotype == Genotype)  %>% filter(age == "18mos")  %>% rownames()
cellphoneDB_bcl6_double_sick<- cellphoneDB_final@meta.data  %>% filter(genotype == Genotype)  %>% filter(age == "sick")  %>% rownames()

cellphoneDB_bcl6_double_6mos_obj<- subset(cellphoneDB_final, cells = cellphoneDB_bcl6_double_6mos)
cellphoneDB_bcl6_double_14mos_obj<- subset(cellphoneDB_final, cells = cellphoneDB_bcl6_double_14mos)
cellphoneDB_bcl6_double_18mos_obj<- subset(cellphoneDB_final, cells = cellphoneDB_bcl6_double_18mos)
cellphoneDB_bcl6_double_sick_obj<- subset(cellphoneDB_final, cells = cellphoneDB_bcl6_double_sick)

saveRDS(object = cellphoneDB_bcl6_double_6mos_obj, file = "./data/cellphoneDB/cellphoneDB_bcl6_double_6mos.obj")
saveRDS(object = cellphoneDB_bcl6_double_14mos_obj, file = "./data/cellphoneDB/cellphonebDB_bcl6_double_14mos.obj")
saveRDS(object = cellphoneDB_bcl6_double_18mos_obj, file = "./data/cellphoneDB/cellphoneDB_bcl6_double_18mos.obj")
saveRDS(object = cellphoneDB_bcl6_double_sick_obj, file = "./data/cellphoneDB/cellphoneDB_bcl6_double_sick.obj")

write10xCounts(x = cellphoneDB_bcl6_double_6mos_obj@assays$RNA@counts, path = "./data/cellphoneDB/10X_format/bcl6_double_6mos/", version = '3')
write10xCounts(x = cellphoneDB_bcl6_double_14mos_obj@assays$RNA@counts, path = "./data/cellphoneDB/10X_format/bcl6_double_14mos/", version = '3')
write10xCounts(x = cellphoneDB_bcl6_double_18mos_obj@assays$RNA@counts, path = "./data/cellphoneDB/10X_format/bcl6_double_18mos/", version = '3')
write10xCounts(x = cellphoneDB_bcl6_double_sick_obj@assays$RNA@counts, path = "./data/cellphoneDB/10X_format/bcl6_double_sick", version = '3')

cellphoneDB_bcl6_double_6mos_meta<- data.frame(Cell = rownames(cellphoneDB_bcl6_double_6mos_obj@meta.data), 
                                        cell_type = cellphoneDB_bcl6_double_6mos_obj@meta.data$new_annotation)
cellphoneDB_bcl6_double_14mos_meta<- data.frame(Cell = rownames(cellphoneDB_bcl6_double_14mos_obj@meta.data), 
                                        cell_type = cellphoneDB_bcl6_double_14mos_obj@meta.data$new_annotation)

cellphoneDB_bcl6_double_18mos_meta<- data.frame(Cell = rownames(cellphoneDB_bcl6_double_18mos_obj@meta.data), 
                                        cell_type = cellphoneDB_bcl6_double_18mos_obj@meta.data$new_annotation)

cellphoneDB_bcl6_double_sick_meta<- data.frame(Cell = rownames(cellphoneDB_bcl6_double_sick_obj@meta.data), 
                                        cell_type = cellphoneDB_bcl6_double_sick_obj@meta.data$new_annotation)

write.table(cellphoneDB_bcl6_double_6mos_meta, file = "./data/cellphoneDB/10X_format/bcl6_double_6mos_meta.txt", sep = "\t", 
            col.names = TRUE, row.names = FALSE, quote = FALSE)
write.table(cellphoneDB_bcl6_double_14mos_meta, file = "./data/cellphoneDB/10X_format/bcl6_double_14mos_meta.txt", sep = "\t", 
            col.names = TRUE, row.names = FALSE, quote = FALSE)
write.table(cellphoneDB_bcl6_double_18mos_meta, file = "./data/cellphoneDB/10X_format/bcl6_double_18mos_meta.txt", sep = "\t", 
            col.names = TRUE, row.names = FALSE, quote = FALSE)
write.table(cellphoneDB_bcl6_double_sick_meta, file = "./data/cellphoneDB/10X_format/bcl6_double_sick_meta.txt", sep = "\t", 
            col.names = TRUE, row.names = FALSE, quote = FALSE)


In [ ]:
Genotype = "WT"
cellphoneDB_wt_6mos<- cellphoneDB_final@meta.data  %>% filter(genotype == Genotype)  %>% filter(age == "6mos")  %>% rownames()
cellphoneDB_wt_14mos<- cellphoneDB_final@meta.data  %>% filter(genotype == Genotype)  %>% filter(age == "14mos")  %>% rownames()
cellphoneDB_wt_18mos<- cellphoneDB_final@meta.data  %>% filter(genotype == Genotype)  %>% filter(age == "18mos")  %>% rownames()

cellphoneDB_wt_6mos_obj<- subset(cellphoneDB_final, cells = cellphoneDB_wt_6mos)
cellphoneDB_wt_14mos_obj<- subset(cellphoneDB_final, cells = cellphoneDB_wt_14mos)
cellphoneDB_wt_18mos_obj<- subset(cellphoneDB_final, cells = cellphoneDB_wt_18mos)

saveRDS(object = cellphoneDB_wt_6mos_obj, file = "./data/cellphoneDB/cellphoneDB_wt_6mos.obj")
saveRDS(object = cellphoneDB_wt_14mos_obj, file = "./data/cellphoneDB/cellphonebDB_wt_14mos.obj")
saveRDS(object = cellphoneDB_wt_18mos_obj, file = "./data/cellphoneDB/cellphoneDB_wt_18mos.obj")

write10xCounts(x = cellphoneDB_wt_6mos_obj@assays$RNA@counts, path = "./data/cellphoneDB/10X_format/wt_6mos/", version = '3')
write10xCounts(x = cellphoneDB_wt_14mos_obj@assays$RNA@counts, path = "./data/cellphoneDB/10X_format/wt_14mos/", version = '3')
write10xCounts(x = cellphoneDB_wt_18mos_obj@assays$RNA@counts, path = "./data/cellphoneDB/10X_format/wt_18mos/", version = '3')

cellphoneDB_wt_6mos_meta<- data.frame(Cell = rownames(cellphoneDB_wt_6mos_obj@meta.data), 
                                        cell_type = cellphoneDB_wt_6mos_obj@meta.data$new_annotation)
cellphoneDB_wt_14mos_meta<- data.frame(Cell = rownames(cellphoneDB_wt_14mos_obj@meta.data), 
                                        cell_type = cellphoneDB_wt_14mos_obj@meta.data$new_annotation)
cellphoneDB_wt_18mos_meta<- data.frame(Cell = rownames(cellphoneDB_wt_18mos_obj@meta.data), 
                                        cell_type = cellphoneDB_wt_18mos_obj@meta.data$new_annotation)

write.table(cellphoneDB_wt_6mos_meta, file = "./data/cellphoneDB/10X_format/wt_6mos_meta.txt", sep = "\t", 
            col.names = TRUE, row.names = FALSE, quote = FALSE)
write.table(cellphoneDB_wt_14mos_meta, file = "./data/cellphoneDB/10X_format/wt_14mos_meta.txt", sep = "\t", 
            col.names = TRUE, row.names = FALSE, quote = FALSE)
write.table(cellphoneDB_wt_18mos_meta, file = "./data/cellphoneDB/10X_format/wt_18mos_meta.txt", sep = "\t", 
            col.names = TRUE, row.names = FALSE, quote = FALSE)

### Run cellphoneDB on each genotype x age combination. The slurm scripts to do this are under 9_cellphoneDB/.
### Read output after running cellphonedb: 

In [6]:
# read bcl6 single
ph_bcl6_6mos_pvals <- read.delim("./data/cellphoneDB/output/bcl6_6mos_out/pvalues.txt", check.names = FALSE)
ph_bcl6_6mos_means <- read.delim("./data/cellphoneDB/output/bcl6_6mos_out/means.txt", check.names = FALSE)
ph_bcl6_14mos_pvals <- read.delim("./data/cellphoneDB/output/bcl6_14mos_out/pvalues.txt", check.names = FALSE)
ph_bcl6_14mos_means <- read.delim("./data/cellphoneDB/output/bcl6_14mos_out/means.txt", check.names = FALSE)
ph_bcl6_18mos_pvals <- read.delim("./data/cellphoneDB/output/bcl6_18mos_out/pvalues.txt", check.names = FALSE)
ph_bcl6_18mos_means <- read.delim("./data/cellphoneDB/output/bcl6_18mos_out/means.txt", check.names = FALSE)
ph_bcl6_sick_pvals <- read.delim("./data/cellphoneDB/output/bcl6_sick_out/pvalues.txt", check.names = FALSE)
ph_bcl6_sick_means <- read.delim("./data/cellphoneDB/output/bcl6_sick_out/means.txt", check.names = FALSE)

# read bcl6 cd70 double
ph_bcl6_double_6mos_pvals <- read.delim("./data/cellphoneDB/output/bcl6_double_6mos_out/pvalues.txt", check.names = FALSE)
ph_bcl6_double_6mos_means <- read.delim("./data/cellphoneDB/output/bcl6_double_6mos_out/means.txt", check.names = FALSE)
ph_bcl6_double_14mos_pvals <- read.delim("./data/cellphoneDB/output/bcl6_double_14mos_out/pvalues.txt", check.names = FALSE)
ph_bcl6_double_14mos_means <- read.delim("./data/cellphoneDB/output/bcl6_double_14mos_out/means.txt", check.names = FALSE)
ph_bcl6_double_18mos_pvals <- read.delim("./data/cellphoneDB/output/bcl6_double_18mos_out/pvalues.txt", check.names = FALSE)
ph_bcl6_double_18mos_means <- read.delim("./data/cellphoneDB/output/bcl6_double_18mos_out/means.txt", check.names = FALSE)
ph_bcl6_double_sick_pvals <- read.delim("./data/cellphoneDB/output/bcl6_double_sick_out/pvalues.txt", check.names = FALSE)
ph_bcl6_double_sick_means <- read.delim("./data/cellphoneDB/output/bcl6_double_sick_out/means.txt", check.names = FALSE)

# read wt
ph_wt_6mos_pvals <- read.delim("./data/cellphoneDB/output/wt_6mos_out/pvalues.txt", check.names = FALSE)
ph_wt_6mos_means <- read.delim("./data/cellphoneDB/output/wt_6mos_out/means.txt", check.names = FALSE)
ph_wt_14mos_pvals <- read.delim("./data/cellphoneDB/output/wt_14mos_out/pvalues.txt", check.names = FALSE)
ph_wt_14mos_means <- read.delim("./data/cellphoneDB/output/wt_14mos_out/means.txt", check.names = FALSE)
ph_wt_18mos_pvals <- read.delim("./data/cellphoneDB/output/wt_18mos_out/pvalues.txt", check.names = FALSE)
ph_wt_18mos_means <- read.delim("./data/cellphoneDB/output/wt_18mos_out/means.txt", check.names = FALSE)

## cd3_pos_cd8_neg_CTLs vs B_cell

In [ ]:
cd3_pos_cd8_neg_CTLs_clusters<- c("cd3_pos_cd8_neg_CTLs")

B_cell_clusters<- c("B_cell_Innate-like", "B_cell_Aged/autoimmune", "B_MHC-II lo Cycling")

In [2]:
cd3_pos_cd8_neg_CTLs_B_cell_interacts<- c()
B_cell_cd3_pos_cd8_neg_CTLs_interacts<- c()

for (c1 in cd3_pos_cd8_neg_CTLs_clusters){
    for (c2 in B_cell_clusters){
        
        interact<- paste0(c1, "|", c2)
        inverse_interact<- paste0(c2, "|", c1)
        
        cd3_pos_cd8_neg_CTLs_B_cell_interacts<- c(cd3_pos_cd8_neg_CTLs_B_cell_interacts, interact)
        B_cell_cd3_pos_cd8_neg_CTLs_interacts<- c(B_cell_cd3_pos_cd8_neg_CTLs_interacts, inverse_interact)
    }
}

cd3_pos_cd8_neg_CTLs_B_cell_interacts
sum(colnames(ph_wt_6mos_pvals) %in% cd3_pos_cd8_neg_CTLs_B_cell_interacts) == length(cd3_pos_cd8_neg_CTLs_B_cell_interacts)

B_cell_cd3_pos_cd8_neg_CTLs_interacts
sum(colnames(ph_wt_6mos_pvals) %in% B_cell_cd3_pos_cd8_neg_CTLs_interacts) == length(B_cell_cd3_pos_cd8_neg_CTLs_interacts)

In [26]:
cd3_pos_cd8_neg_B_cell_mtx_bcl6 = build_cpdb_mtx(cols = cd3_pos_cd8_neg_CTLs_B_cell_interacts,
                                            scaled = FALSE,
                                            sig.cellpairs = 3,
                                            sig.timepoint = 1,  
                                            group_name = "_bcl6",
                                            ph_6mos_pvals = ph_bcl6_6mos_pvals,
                                            ph_6mos_means = ph_bcl6_6mos_means,
                                            ph_14mos_pvals = ph_bcl6_14mos_pvals,
                                            ph_14mos_means = ph_bcl6_14mos_means,
                                            ph_18mos_pvals = ph_bcl6_18mos_pvals,
                                            ph_18mos_means = ph_bcl6_18mos_means,
                                            ph_sick_pvals = ph_bcl6_sick_pvals,
                                            ph_sick_means = ph_bcl6_sick_means)

B_cell_cd3_pos_cd8_neg_mtx_bcl6 = build_cpdb_mtx(cols = B_cell_cd3_pos_cd8_neg_CTLs_interacts, 
                                            scaled = FALSE,
                                            sig.cellpairs = 3,
                                            sig.timepoint = 1,  
                                            group_name = "_bcl6",
                                            ph_6mos_pvals = ph_bcl6_6mos_pvals,
                                            ph_6mos_means = ph_bcl6_6mos_means,
                                            ph_14mos_pvals = ph_bcl6_14mos_pvals,
                                            ph_14mos_means = ph_bcl6_14mos_means,
                                            ph_18mos_pvals = ph_bcl6_18mos_pvals,
                                            ph_18mos_means = ph_bcl6_18mos_means,
                                            ph_sick_pvals = ph_bcl6_sick_pvals,
                                            ph_sick_means = ph_bcl6_sick_means)

cd3_pos_cd8_neg_B_cell_mtx_bcl6_double = build_cpdb_mtx(cols = cd3_pos_cd8_neg_CTLs_B_cell_interacts,
                                            scaled = FALSE,
                                            group_name = "_bcl6_double",
                                            sig.cellpairs = 3,
                                            sig.timepoint = 1, 
                                            ph_6mos_pvals = ph_bcl6_double_6mos_pvals,
                                            ph_6mos_means = ph_bcl6_double_6mos_means,
                                            ph_14mos_pvals = ph_bcl6_double_14mos_pvals,
                                            ph_14mos_means = ph_bcl6_double_14mos_means,
                                            ph_18mos_pvals = ph_bcl6_double_18mos_pvals,
                                            ph_18mos_means = ph_bcl6_double_18mos_means,
                                            ph_sick_pvals = ph_bcl6_double_sick_pvals,
                                            ph_sick_means = ph_bcl6_double_sick_means)

B_cell_cd3_pos_cd8_neg_mtx_bcl6_double = build_cpdb_mtx(cols = B_cell_cd3_pos_cd8_neg_CTLs_interacts, 
                                            scaled = FALSE,
                                            group_name = "_bcl6_double",
                                            sig.cellpairs = 3,
                                            sig.timepoint = 1, 
                                            ph_6mos_pvals = ph_bcl6_double_6mos_pvals,
                                            ph_6mos_means = ph_bcl6_double_6mos_means,
                                            ph_14mos_pvals = ph_bcl6_double_14mos_pvals,
                                            ph_14mos_means = ph_bcl6_double_14mos_means,
                                            ph_18mos_pvals = ph_bcl6_double_18mos_pvals,
                                            ph_18mos_means = ph_bcl6_double_18mos_means,
                                            ph_sick_pvals = ph_bcl6_double_sick_pvals,
                                            ph_sick_means = ph_bcl6_double_sick_means)

cd3_pos_cd8_neg_B_cell_mtx_wt = build_cpdb_mtx_wt(cols = cd3_pos_cd8_neg_CTLs_B_cell_interacts,
                                            scaled = FALSE,
                                            group_name = "_wt",
                                            sig.cellpairs = 3,
                                            sig.timepoint = 1, 
                                            ph_6mos_pvals = ph_wt_6mos_pvals,
                                            ph_6mos_means = ph_wt_6mos_means,
                                            ph_14mos_pvals = ph_wt_14mos_pvals,
                                            ph_14mos_means = ph_wt_14mos_means,
                                            ph_18mos_pvals = ph_wt_18mos_pvals,
                                            ph_18mos_means = ph_wt_18mos_means)

B_cell_cd3_pos_cd8_neg_mtx_wt = build_cpdb_mtx_wt(cols = B_cell_cd3_pos_cd8_neg_CTLs_interacts, 
                                            scaled = FALSE,
                                            group_name = "_wt",
                                            sig.cellpairs = 3,
                                            sig.timepoint = 1, 
                                            ph_6mos_pvals = ph_wt_6mos_pvals,
                                            ph_6mos_means = ph_wt_6mos_means,
                                            ph_14mos_pvals = ph_wt_14mos_pvals,
                                            ph_14mos_means = ph_wt_14mos_means,
                                            ph_18mos_pvals = ph_wt_18mos_pvals,
                                            ph_18mos_means = ph_wt_18mos_means)


cd3_pos_cd8_neg_CTLs_B_cell_mtx<- merge_mtx(cd3_pos_cd8_neg_B_cell_mtx_wt, cd3_pos_cd8_neg_B_cell_mtx_bcl6, cd3_pos_cd8_neg_B_cell_mtx_bcl6_double)
B_cell_cd3_pos_cd8_neg_CTLs_mtx<- merge_mtx(B_cell_cd3_pos_cd8_neg_mtx_wt, B_cell_cd3_pos_cd8_neg_mtx_bcl6, B_cell_cd3_pos_cd8_neg_mtx_bcl6_double)

hp1 = plt_cpdb_dotplot_with_sick(exp_mtx = cd3_pos_cd8_neg_CTLs_B_cell_mtx$exp_mtx, 
                       pval_mtx = cd3_pos_cd8_neg_CTLs_B_cell_mtx$pval_mtx)
hp2 = plt_cpdb_dotplot_with_sick(exp_mtx = B_cell_cd3_pos_cd8_neg_CTLs_mtx$exp_mtx, 
                       pval_mtx = B_cell_cd3_pos_cd8_neg_CTLs_mtx$pval_mtx)

pdf("./results/figures/9_Dotplot_cd3_pos_cd8_neg_CTLs_B_cell_wt_bcl6_and_bcl6_double_cell3_time1_genotype1.pdf", width = 12, height = 10)
draw(hp1, annotation_legend_list = lgd, ht_gap = unit(1, "cm"))
dev.off()

pdf("./results/figures/9_Dotplot_B_cell_cd3_pos_cd8_neg_CTLs_wt_bcl6_and_bcl6_double_cell3_time1_genotype1.pdf", width = 12, height = 10)
draw(hp2, annotation_legend_list = lgd, ht_gap = unit(1, "cm"))
dev.off()

total number of intersections considered: 26

6 mos...

14 mos...

18 mos...

sick...

Generating count matrix...

Generating pvalue matrix...

output unscaled data

total number of intersections considered: 23

6 mos...

14 mos...

18 mos...

sick...

Generating count matrix...

Generating pvalue matrix...

output unscaled data

total number of intersections considered: 20

6 mos...

14 mos...

18 mos...

sick...

Generating count matrix...

Generating pvalue matrix...

output unscaled data

total number of intersections considered: 20

6 mos...

14 mos...

18 mos...

sick...

Generating count matrix...

Generating pvalue matrix...

output unscaled data

total number of intersections considered: 21

6 mos...

14 mos...

18 mos...

Generating count matrix...

Generating pvalue matrix...

output unscaled data

total number of intersections considered: 19

6 mos...

14 mos...

18 mos...

Generating count matrix...

Generating pvalue matrix...

output unscaled data



In [33]:
cd3_pos_cd8_neg_B_cell_mtx_bcl6 = build_cpdb_mtx(cols = cd3_pos_cd8_neg_CTLs_B_cell_interacts,
                                            scaled = FALSE,
                                            sig.cellpairs = 1,
                                            sig.timepoint = 1,  
                                            group_name = "_bcl6",
                                            ph_6mos_pvals = ph_bcl6_6mos_pvals,
                                            ph_6mos_means = ph_bcl6_6mos_means,
                                            ph_14mos_pvals = ph_bcl6_14mos_pvals,
                                            ph_14mos_means = ph_bcl6_14mos_means,
                                            ph_18mos_pvals = ph_bcl6_18mos_pvals,
                                            ph_18mos_means = ph_bcl6_18mos_means,
                                            ph_sick_pvals = ph_bcl6_sick_pvals,
                                            ph_sick_means = ph_bcl6_sick_means)

B_cell_cd3_pos_cd8_neg_mtx_bcl6 = build_cpdb_mtx(cols = B_cell_cd3_pos_cd8_neg_CTLs_interacts, 
                                            scaled = FALSE,
                                            sig.cellpairs = 1,
                                            sig.timepoint = 1,  
                                            group_name = "_bcl6",
                                            ph_6mos_pvals = ph_bcl6_6mos_pvals,
                                            ph_6mos_means = ph_bcl6_6mos_means,
                                            ph_14mos_pvals = ph_bcl6_14mos_pvals,
                                            ph_14mos_means = ph_bcl6_14mos_means,
                                            ph_18mos_pvals = ph_bcl6_18mos_pvals,
                                            ph_18mos_means = ph_bcl6_18mos_means,
                                            ph_sick_pvals = ph_bcl6_sick_pvals,
                                            ph_sick_means = ph_bcl6_sick_means)

cd3_pos_cd8_neg_B_cell_mtx_bcl6_double = build_cpdb_mtx(cols = cd3_pos_cd8_neg_CTLs_B_cell_interacts,
                                            scaled = FALSE,
                                            group_name = "_bcl6_double",
                                            sig.cellpairs = 1,
                                            sig.timepoint = 1, 
                                            ph_6mos_pvals = ph_bcl6_double_6mos_pvals,
                                            ph_6mos_means = ph_bcl6_double_6mos_means,
                                            ph_14mos_pvals = ph_bcl6_double_14mos_pvals,
                                            ph_14mos_means = ph_bcl6_double_14mos_means,
                                            ph_18mos_pvals = ph_bcl6_double_18mos_pvals,
                                            ph_18mos_means = ph_bcl6_double_18mos_means,
                                            ph_sick_pvals = ph_bcl6_double_sick_pvals,
                                            ph_sick_means = ph_bcl6_double_sick_means)

B_cell_cd3_pos_cd8_neg_mtx_bcl6_double = build_cpdb_mtx(cols = B_cell_cd3_pos_cd8_neg_CTLs_interacts, 
                                            scaled = FALSE,
                                            group_name = "_bcl6_double",
                                            sig.cellpairs = 1,
                                            sig.timepoint = 1, 
                                            ph_6mos_pvals = ph_bcl6_double_6mos_pvals,
                                            ph_6mos_means = ph_bcl6_double_6mos_means,
                                            ph_14mos_pvals = ph_bcl6_double_14mos_pvals,
                                            ph_14mos_means = ph_bcl6_double_14mos_means,
                                            ph_18mos_pvals = ph_bcl6_double_18mos_pvals,
                                            ph_18mos_means = ph_bcl6_double_18mos_means,
                                            ph_sick_pvals = ph_bcl6_double_sick_pvals,
                                            ph_sick_means = ph_bcl6_double_sick_means)

cd3_pos_cd8_neg_B_cell_mtx_wt = build_cpdb_mtx_wt(cols = cd3_pos_cd8_neg_CTLs_B_cell_interacts,
                                            scaled = FALSE,
                                            group_name = "_wt",
                                            sig.cellpairs = 1,
                                            sig.timepoint = 1, 
                                            ph_6mos_pvals = ph_wt_6mos_pvals,
                                            ph_6mos_means = ph_wt_6mos_means,
                                            ph_14mos_pvals = ph_wt_14mos_pvals,
                                            ph_14mos_means = ph_wt_14mos_means,
                                            ph_18mos_pvals = ph_wt_18mos_pvals,
                                            ph_18mos_means = ph_wt_18mos_means)

B_cell_cd3_pos_cd8_neg_mtx_wt = build_cpdb_mtx_wt(cols = B_cell_cd3_pos_cd8_neg_CTLs_interacts, 
                                            scaled = FALSE,
                                            group_name = "_wt",
                                            sig.cellpairs = 1,
                                            sig.timepoint = 1, 
                                            ph_6mos_pvals = ph_wt_6mos_pvals,
                                            ph_6mos_means = ph_wt_6mos_means,
                                            ph_14mos_pvals = ph_wt_14mos_pvals,
                                            ph_14mos_means = ph_wt_14mos_means,
                                            ph_18mos_pvals = ph_wt_18mos_pvals,
                                            ph_18mos_means = ph_wt_18mos_means)


cd3_pos_cd8_neg_CTLs_B_cell_mtx<- merge_mtx(cd3_pos_cd8_neg_B_cell_mtx_wt, cd3_pos_cd8_neg_B_cell_mtx_bcl6, cd3_pos_cd8_neg_B_cell_mtx_bcl6_double)
B_cell_cd3_pos_cd8_neg_CTLs_mtx<- merge_mtx(B_cell_cd3_pos_cd8_neg_mtx_wt, B_cell_cd3_pos_cd8_neg_mtx_bcl6, B_cell_cd3_pos_cd8_neg_mtx_bcl6_double)

hp1 = plt_cpdb_dotplot_with_sick(exp_mtx = cd3_pos_cd8_neg_CTLs_B_cell_mtx$exp_mtx, 
                       pval_mtx = cd3_pos_cd8_neg_CTLs_B_cell_mtx$pval_mtx)
hp2 = plt_cpdb_dotplot_with_sick(exp_mtx = B_cell_cd3_pos_cd8_neg_CTLs_mtx$exp_mtx, 
                       pval_mtx = B_cell_cd3_pos_cd8_neg_CTLs_mtx$pval_mtx)

pdf("./results/figures/9_Dotplot_cd3_pos_cd8_neg_CTLs_B_cell_wt_bcl6_and_bcl6_double_cell1_time1_genotype1.pdf", width = 12, height = 10)
draw(hp1, annotation_legend_list = lgd, ht_gap = unit(1, "cm"))
dev.off()

pdf("./results/figures/9_Dotplot_B_cell_cd3_pos_cd8_neg_CTLs_wt_bcl6_and_bcl6_double_cell1_time1_genotype1.pdf", width = 12, height = 10)
draw(hp2, annotation_legend_list = lgd, ht_gap = unit(1, "cm"))
dev.off()

total number of intersections considered: 40

6 mos...

14 mos...

18 mos...

sick...

Generating count matrix...

Generating pvalue matrix...

output unscaled data

total number of intersections considered: 48

6 mos...

14 mos...

18 mos...

sick...

Generating count matrix...

Generating pvalue matrix...

output unscaled data

total number of intersections considered: 40

6 mos...

14 mos...

18 mos...

sick...

Generating count matrix...

Generating pvalue matrix...

output unscaled data

total number of intersections considered: 45

6 mos...

14 mos...

18 mos...

sick...

Generating count matrix...

Generating pvalue matrix...

output unscaled data

total number of intersections considered: 44

6 mos...

14 mos...

18 mos...

Generating count matrix...

Generating pvalue matrix...

output unscaled data

total number of intersections considered: 38

6 mos...

14 mos...

18 mos...

Generating count matrix...

Generating pvalue matrix...

output unscaled data

wt...

bcl6...

bcl6 d

[1] NA NA


TRUE

scaling the count matrix

wt...

bcl6...

bcl6 double...



[1] NA NA


TRUE

scaling the count matrix

quantile: -1.1087756153354-0.3253213656421291.91819397336388

12129

9996

quantile: -1.16205336670373-0.183710169412131.79918673420415

12129

9996



png 
  2

png 
  2